# B5 · Modèles de diffusion : un DDPM minimal — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b5_diffusion/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B5.Q1 à B5.Q11, B5.R1 à B5.R3, B5.1 à B5.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B5.12 à B5.18 | les mathématiques de ta librairie et leurs contrôles : plannings de bruit, bruitage en forme fermée, posterior, pas DDPM et DDIM, aller-retour avec le vrai bruit (*prochaine session de génération*) | | | |
| B | B5.19 et B5.20 | jouets 2D : un MLP débruiteur conditionné par le pas, puis le flow matching (*prochaine session de génération*) | | | |
| C | B5.21 à B5.31 | le U-Net sur MNIST et Fashion-MNIST : quatre bugs, entraînement avec EMA, DDPM contre DDIM, guidance, mémorisation, distance de Fréchet, script autonome et défi (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Dériver la forme fermée du bruitage et le posterior d'un pas, et calculer à la main un pas DDPM et un pas DDIM.
- Implémenter en NumPy les plannings de bruit, le bruitage et les pas d'échantillonnage, et les contrôler par l'aller-retour avec le vrai bruit.
- Écrire et entraîner un petit U-Net conditionné par le pas, l'échantillonner par DDPM et DDIM, et le conditionner par la classe avec la guidance sans classifieur.
- Évaluer un générateur (distance de Fréchet, plus proches voisins) et le situer face au VAE et au GAN.

**Rappel express.** Pas de l'article $t = 1..T$, indice du code $t - 1$. $\alpha_t = 1 - \beta_t$, $\bar\alpha_t = \prod_{s \le t} \alpha_s$, $\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1 - \bar\alpha_t}\,\boldsymbol{\varepsilon}$, $\operatorname{SNR}(t) = \bar\alpha_t / (1 - \bar\alpha_t)$. Posterior : $\tilde\beta_t = \frac{1 - \bar\alpha_{t-1}}{1 - \bar\alpha_t}\beta_t$. DDPM : $\mathbf{x}_{t-1} = \frac{1}{\sqrt{\alpha_t}}\big(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1 - \bar\alpha_t}}\boldsymbol{\varepsilon}_\theta\big) + \sigma_t\mathbf{z}$, sans bruit au dernier pas. DDIM ($\eta = 0$) : $\hat{\mathbf{x}}_0 = (\mathbf{x}_t - \sqrt{1 - \bar\alpha_t}\,\boldsymbol{\varepsilon}_\theta) / \sqrt{\bar\alpha_t}$, puis $\mathbf{x}_{t'} = \sqrt{\bar\alpha_{t'}}\,\hat{\mathbf{x}}_0 + \sqrt{1 - \bar\alpha_{t'}}\,\boldsymbol{\varepsilon}_\theta$. Guidance : $\tilde{\boldsymbol{\varepsilon}} = (1 + w)\,\boldsymbol{\varepsilon}_\theta(\mathbf{x}_t, c) - w\,\boldsymbol{\varepsilon}_\theta(\mathbf{x}_t, \varnothing)$, `guidance_scale` $= 1 + w$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B5.1 à B5.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; un vecteur : une liste de nombres (`[0.5, -1.25]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ B5.6 et B5.7, l'estimation 🧮 B5.8, la réflexion (🗣️ B5.9, ⚖️ B5.10, 📄 B5.11) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md). Steps: t = 1..T in the
# formulas (Ho et al.), indices 0..T-1 in the code (step t is index t - 1).
import math

import numpy as np

# ✏️ B5.1: classifier-free guidance on two pixels, (1 + w) eps_c - w eps_uncond
EC_B51, EU_B51 = np.array([0.8, -0.5]), np.array([0.5, 0.1])


def cfg_b51(w, cond=EC_B51, uncond=EU_B51):
    return [float(v) for v in (1 + w) * cond - w * uncond]


W0_B51, W1_B51, W3_B51 = cfg_b51(0), cfg_b51(1), cfg_b51(3)
GAP_B51 = 3 * float(np.linalg.norm(EC_B51 - EU_B51))                # || eps_tilde - eps_c || for w = 3
EVALS_B51 = 2 * 50                                                  # two predictions per step, 50 DDIM steps

# ✏️ B5.2: toy linear schedule T = 4, beta from 0.05 to 0.35 (it serves B5.4 to B5.7 too)
BETAS_B52 = np.linspace(0.05, 0.35, 4)
ALPHAS_B52 = 1 - BETAS_B52
AB_B52 = np.cumprod(ALPHAS_B52)                                     # alpha_bar, indices 0..3 (steps 1..4)
BETA2_B52, AB2_B52, AB4_B52 = float(BETAS_B52[1]), float(AB_B52[1]), float(AB_B52[3])
KEEP4_B52 = math.sqrt(AB4_B52)
APPROX4_B52 = math.exp(-float(BETAS_B52.sum()))
SUM1000_B52 = 1000 * (1e-4 + 0.02) / 2                              # arithmetic series: T (beta_1 + beta_T) / 2
AB1000_E5_B52 = math.exp(-SUM1000_B52) * 1e5
KEEP1000_PCT_B52 = 100 * math.sqrt(AB1000_E5_B52 * 1e-5)
SUM400_B52 = 400 * (1e-4 + 0.02) / 2
SUM400_SCALED_B52 = 400 * (1e-4 + 0.02) * (1000 / 400) / 2

# ✏️ B5.3: signal-to-noise ratio of the linear schedule (approximation exp(-S(t))) and of the cosine schedule (exact)
T_B53, B1_B53, BT_B53, S_B53 = 1000, 1e-4, 0.02, 0.008


def lin_sum_b53(t, square=False):
    """S(t), the sum of the first t betas; square=True: the t**2 slip instead of t (t - 1)."""
    return t * B1_B53 + (BT_B53 - B1_B53) * (t * t if square else t * (t - 1)) / (2 * (T_B53 - 1))


def snr_b53(ab):
    return ab / (1 - ab)


def cos_ab_b53(u, s=S_B53):
    f = lambda v: math.cos((v + s) / (1 + s) * math.pi / 2) ** 2
    return f(u) / f(0)


def lin_cross_b53(target_sum):
    """The t (real) where S(t) = target_sum: a t**2 + b t - target_sum = 0, then the fraction t / T."""
    a = (BT_B53 - B1_B53) / (2 * (T_B53 - 1))
    b = B1_B53 - a
    return (-b + math.sqrt(b * b + 4 * a * target_sum)) / (2 * a) / T_B53


def cos_cross_b53(alpha_bar, s=S_B53):
    """The fraction u where the cosine alpha_bar(u) equals alpha_bar."""
    f0 = math.cos(s / (1 + s) * math.pi / 2) ** 2
    theta = math.acos(math.sqrt(alpha_bar * f0))
    return theta * 2 / math.pi * (1 + s) - s


S500_B53 = lin_sum_b53(500)
LIN500_B53 = snr_b53(math.exp(-S500_B53))
COS500_B53 = snr_b53(cos_ab_b53(0.5))
LIN1_B53, COS1_B53 = lin_cross_b53(math.log(2)), cos_cross_b53(0.5)           # SNR = 1: alpha_bar = 1/2
LIN01_B53, COS01_B53 = lin_cross_b53(math.log(11)), cos_cross_b53(1 / 11)     # SNR = 0.1: alpha_bar = 1/11

# ✏️ B5.4: one DDPM step at step t = 3 (index 2) of the toy schedule
I_B54, X3_B54, EPS_B54, Z_B54 = 2, 0.8, 0.5, -1.0


def ddpm_mean_b54(x, eps, i=I_B54, sign=-1.0, root=True):
    """Mean of Algorithm 2; sign=+1: the sign slip; root=False: beta / (1 - alpha_bar) without the square root."""
    ab = float(AB_B52[i])
    coef = BETAS_B52[i] / (math.sqrt(1 - ab) if root else (1 - ab))
    return float((x + sign * coef * eps) / math.sqrt(ALPHAS_B52[i]))


def posterior_b54(i=I_B54):
    """coef1, coef2 and beta tilde of the posterior at index i (alpha_bar_prev = 1 at index 0)."""
    ab, ab_prev = float(AB_B52[i]), (float(AB_B52[i - 1]) if i > 0 else 1.0)
    beta = float(BETAS_B52[i])
    return (beta * math.sqrt(ab_prev) / (1 - ab), (1 - ab_prev) * math.sqrt(ALPHAS_B52[i]) / (1 - ab),
            beta * (1 - ab_prev) / (1 - ab))


COEF_B54 = float(BETAS_B52[I_B54] / math.sqrt(1 - AB_B52[I_B54]))
MEAN_B54 = ddpm_mean_b54(X3_B54, EPS_B54)
C1_B54, C2_B54, BTILDE_B54 = posterior_b54()
X2_B54 = MEAN_B54 + math.sqrt(BTILDE_B54) * Z_B54
X2_BETA_B54 = MEAN_B54 + math.sqrt(BETAS_B52[I_B54]) * Z_B54
X3C_B54, EPSC_B54 = 1.5, -0.6
X0HAT_C_B54 = (X3C_B54 - math.sqrt(1 - AB_B52[I_B54]) * EPSC_B54) / math.sqrt(AB_B52[I_B54])
CLIP_MEAN_B54 = C1_B54 * float(np.clip(X0HAT_C_B54, -1, 1)) + C2_B54 * X3C_B54
NOCLIP_MEAN_B54 = C1_B54 * X0HAT_C_B54 + C2_B54 * X3C_B54
LAST_B54 = ddpm_mean_b54(0.5, 0.2, i=0)                                         # step 1 (index 0): no noise

# ✏️ B5.5: DDIM from step 4 (index 3), x_4 = -0.5, eps_theta = -1.2
X4_B55, EPS_B55 = -0.5, -1.2


def x0hat_b55(x=X4_B55, eps=EPS_B55, i=3):
    return float((x - math.sqrt(1 - AB_B52[i]) * eps) / math.sqrt(AB_B52[i]))


def ab_b55(i):
    return 1.0 if i == -1 else float(AB_B52[i])


def ddim_b55(i, i_prev, eta=0.0, x=X4_B55, eps=EPS_B55, noise_term=True):
    """The deterministic part of a DDIM step (z = 0), and sigma."""
    a, ap = ab_b55(i), ab_b55(i_prev)
    sigma = eta * math.sqrt((1 - ap) / (1 - a)) * math.sqrt(1 - a / ap)
    x0 = (x - math.sqrt(1 - a) * eps) / math.sqrt(a)
    return math.sqrt(ap) * x0 + (math.sqrt(1 - ap - sigma ** 2) * eps if noise_term else 0.0), sigma


X0HAT_B55 = x0hat_b55()
X1_B55 = ddim_b55(3, 0)[0]
X2_B55 = ddim_b55(3, 1)[0]
SIGMA2_43_B55 = ddim_b55(3, 2, eta=1.0)[1] ** 2
SIGMA_41_B55 = ddim_b55(3, 0, eta=1.0)[1]
DIR_41_B55 = math.sqrt(1 - AB_B52[0] - SIGMA_41_B55 ** 2)

# quizzes and recalls with a computation
Q1E = 1000 * 0.02
Q2B = float(np.mean([-1, 0, 2]))
Q2C = float(np.mean((np.array([-1, 0, 2]) - Q2B) ** 2))
Q3A, Q3B, Q3D = math.sqrt(0.64), math.sqrt(1 - 0.64), 0.64 / 0.36
Q4B = (0.5 - math.sqrt(1 - 0.64) * 0.2) / math.sqrt(0.64)
Q5C = 128 // 2
Q6B, Q6D = 128 + 64, 32 // 2 ** 3
Q7A = 1000 * 0.05
Q9A = 3 * 0.3 - 2 * 0.5
Q11A = (256 // 4) ** 2 * 3
R1B = 2 * math.pi
R2A = (0 - 1) ** 2 + (1 + 4 - 2 * math.sqrt(1 * 4))
R2E = 0 ** 2 + (1 + 0 - 2 * math.sqrt(1 * 0))
R3A = 0.5 + math.exp(0.5 * math.log(0.25)) * (-2)
R3B = 0.5 * (0.5 ** 2 + 0.25 - 1 - math.log(0.25))

# The same answers from the libraries (no download): mylearn (reference) and, if installed, diffusers
D = mylearn.diffusion
toy = D.noise_schedule(D.linear_beta_schedule(4, 0.05, 0.35))
assert np.allclose(toy["betas"], BETAS_B52) and np.allclose(toy["alpha_bars"], AB_B52)

# ✏️ B5.2: the sums of the betas, and the exact alpha_bar_T that the approximation overestimates
lin1000 = D.noise_schedule(D.linear_beta_schedule(1000))
assert math.isclose(lin1000["betas"].sum(), SUM1000_B52)
assert math.isclose(D.linear_beta_schedule(400).sum(), SUM400_B52)
assert math.isclose(D.linear_beta_schedule(400, 1e-4 * 2.5, 0.02 * 2.5).sum(), SUM400_SCALED_B52)
assert lin1000["alpha_bars"][-1] < AB1000_E5_B52 * 1e-5 and toy["alpha_bars"][-1] < APPROX4_B52

# ✏️ B5.3: the exact schedules of the library (index i = step i + 1) cross the thresholds at the same 2 decimals
cos1000 = D.noise_schedule(D.cosine_beta_schedule(1000))
for sched, answers in ((lin1000, (LIN1_B53, LIN01_B53)), (cos1000, (COS1_B53, COS01_B53))):
    snr = sched["alpha_bars"] / (1 - sched["alpha_bars"])
    for threshold, answer in zip((1.0, 0.1), answers):
        first_step = int(np.argmax(snr < threshold)) + 1
        assert round(first_step / 1000, 2) == round(answer, 2), (threshold, first_step, answer)
assert math.isclose(cos1000["alpha_bars"][499], cos_ab_b53(0.5), rel_tol=1e-9)
assert round(float(lin1000["alpha_bars"][499] / (1 - lin1000["alpha_bars"][499])), 2) == round(LIN500_B53, 2)

# ✏️ B5.4: ddpm_step; its mean is step(x) - step(0) with the same seed, its variance posterior_variance
def mean_of_ddpm_step(x, i, eps, **kw):
    one = D.ddpm_step(np.array([[x]]), i, np.array([[eps]]), toy, rng=np.random.default_rng(0), **kw)
    zero = D.ddpm_step(np.array([[0.0]]), i, np.array([[0.0]]), toy, rng=np.random.default_rng(0),
                       variance=kw.get("variance", "posterior"))
    return float((one - zero)[0, 0])


assert math.isclose(mean_of_ddpm_step(X3_B54, I_B54, EPS_B54), MEAN_B54)
assert math.isclose(mean_of_ddpm_step(X3_B54, I_B54, EPS_B54, variance="beta"), MEAN_B54)
assert math.isclose(mean_of_ddpm_step(X3C_B54, I_B54, EPSC_B54, clip_x0=1.0), CLIP_MEAN_B54)
assert math.isclose(toy["posterior_variance"][I_B54], BTILDE_B54)
assert math.isclose(toy["posterior_mean_coef1"][I_B54], C1_B54) and math.isclose(toy["posterior_mean_coef2"][I_B54], C2_B54)
for variance in ("posterior", "beta"):                                 # step 1 (index 0): no noise, whatever the seed
    assert math.isclose(float(D.ddpm_step(np.array([[0.5]]), 0, np.array([[0.2]]), toy, rng=7, variance=variance)[0, 0]),
                        LAST_B54)

# ✏️ B5.5: predict_x0_from_eps and ddim_step (eta = 0: deterministic)
assert math.isclose(float(D.predict_x0_from_eps(np.array([X4_B55]), [3], np.array([EPS_B55]), AB_B52)[0]), X0HAT_B55)
x4 = np.array([[X4_B55]]); e4 = np.array([[EPS_B55]])
assert math.isclose(float(D.ddim_step(x4, 3, 0, e4, AB_B52)[0, 0]), X1_B55)
assert math.isclose(float(D.ddim_step(x4, 3, 1, e4, AB_B52)[0, 0]), X2_B55)
assert math.isclose(float(D.ddim_step(D.ddim_step(x4, 3, 1, e4, AB_B52), 1, 0, e4, AB_B52)[0, 0]), X1_B55)
assert math.isclose(float(D.ddim_step(x4, 3, -1, e4, AB_B52)[0, 0]), X0HAT_B55)
assert math.isclose(SIGMA2_43_B55, toy["posterior_variance"][3])

# recalls: positional_encoding (B3), frechet_distance (ch. 27), reparameterize and gaussian_kl (ch. 25)
pe = mylearn.attention.positional_encoding(1000, 128)
assert pe[0, 1] == 1.0 and pe.shape[1] // 2 == Q5C
assert np.allclose(pe[:, 0], np.sin(np.arange(1000)))                 # column 0: sin(p), period 2 pi
assert 2 * math.pi * 10000 ** (126 / 128) > 1000                      # the slowest pair: less than one turn
fd = mylearn.gen_metrics.frechet_distance
assert math.isclose(fd([0.0], [[1.0]], [1.0], [[4.0]]), R2A) and math.isclose(fd([0.0], [[1.0]], [0.0], [[0.0]]), R2E)
z, _ = mylearn.vae.reparameterize([[0.5]], [[math.log(0.25)]], rng=0)
assert z.shape == (1, 1)
assert math.isclose(float(mylearn.vae.gaussian_kl([[0.5]], [[math.log(0.25)]])[0]), R3B)
print("✅ the paper answers agree with mylearn (reference): diffusion, attention, gen_metrics, vae")

# Optional: diffusers' schedulers on the toy schedule (float64 tensors, noise set to zero)
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm, imported by diffusers
try:
    import torch
    import diffusers.schedulers.scheduling_ddpm as ddpm_module
    from diffusers import DDIMScheduler, DDPMScheduler
except ImportError:
    print("ℹ️ diffusers is not installed: the scheduler comparison is skipped (the closed forms above are checked)")
else:
    def as_double(scheduler, names):
        for name in names:
            setattr(scheduler, name, getattr(scheduler, name).double())
        return scheduler

    def diffusers_ddpm(x, i, eps, clip=None):
        sch = as_double(DDPMScheduler(num_train_timesteps=4, trained_betas=BETAS_B52.tolist(),
                                      clip_sample=clip is not None, clip_sample_range=clip or 1.0),
                        ("betas", "alphas", "alphas_cumprod", "one"))
        saved = ddpm_module.randn_tensor
        ddpm_module.randn_tensor = lambda shape, **kw: torch.zeros(shape, dtype=torch.float64)
        try:
            out = sch.step(torch.tensor([[eps]], dtype=torch.float64), i, torch.tensor([[x]], dtype=torch.float64))
        finally:
            ddpm_module.randn_tensor = saved
        return float(out.prev_sample), float(sch._get_variance(i))

    def diffusers_ddim(x, i, eps, n_steps):
        sch = as_double(DDIMScheduler(num_train_timesteps=4, trained_betas=BETAS_B52.tolist(), clip_sample=False,
                                      set_alpha_to_one=True), ("alphas_cumprod", "final_alpha_cumprod"))
        sch.set_timesteps(n_steps)
        out = sch.step(torch.tensor([[eps]], dtype=torch.float64), i, torch.tensor([[x]], dtype=torch.float64),
                       eta=0.0)
        return float(out.prev_sample), float(out.pred_original_sample), sch

    mean, var = diffusers_ddpm(X3_B54, I_B54, EPS_B54)
    assert math.isclose(mean, MEAN_B54, rel_tol=1e-6) and math.isclose(var, BTILDE_B54, rel_tol=1e-6)
    assert math.isclose(diffusers_ddpm(X3C_B54, I_B54, EPSC_B54, clip=1.0)[0], CLIP_MEAN_B54, rel_tol=1e-6)
    assert math.isclose(diffusers_ddpm(0.5, 0, 0.2)[0], LAST_B54, rel_tol=1e-6)
    prev, x0_pred, ddim = diffusers_ddim(X4_B55, 3, EPS_B55, 2)          # 4 // 2 = 2: index 3 -> index 1
    assert math.isclose(prev, X2_B55, rel_tol=1e-6) and math.isclose(x0_pred, X0HAT_B55, rel_tol=1e-6)
    assert math.isclose(diffusers_ddim(X4_B55, 3, EPS_B55, 1)[0], X0HAT_B55, rel_tol=1e-6)   # index 3 -> clean
    assert math.isclose(float(ddim._get_variance(3, 2)), SIGMA2_43_B55, rel_tol=1e-6)
    assert math.isclose(float(ddim._get_variance(3, 0)), SIGMA_41_B55 ** 2, rel_tol=1e-6)
    print("✅ ✏️ B5.4 and B5.5 also agree with diffusers' DDPMScheduler and DDIMScheduler")

✅ the paper answers agree with mylearn (reference): diffusion, attention, gen_metrics, vae


✅ ✏️ B5.4 and B5.5 also agree with diffusers' DDPMScheduler and DDIMScheduler


**B5.Q1 — Diffusion, VAE, GAN : comment chacun fabrique-t-il une image ?**

In [3]:
wb.record("B5.Q1a", "A", mistakes={"relis l'encadré 🔗 « Lien avec les ch. 25 et 27 » : combien de passes du décodeur pour générer avec un VAE ?": "B", "pour générer, part-on d'une image du dataset ? Relis l'encadré 🔗 de 205.1": "C"})
wb.record("B5.Q1b", False, mistakes={"relis « Détruire est facile » (§205.1) : le processus direct s'apprend-il ?": True})
wb.record("B5.Q1c", "C", mistakes={"relis l'encadré 🔗 de 205.1 : la diffusion s'entraîne-t-elle contre un adversaire ?": "A", "le VAE a-t-il un discriminateur ? Relis l'encadré 🔗 de 205.1": "B"})
wb.record("B5.Q1d", True, mistakes={"relis l'encadré 🔗 « Lien avec les ch. 25 et 27 » : quelle taille ont les codes de la diffusion ?": False})
wb.record("B5.Q1e", round(Q1E), mistakes={"c'est le nombre de passes : on demande des secondes": 1000,
          "vérifie la multiplication par 1 000": 2})

WB_ANSWER {"hash": "41720f303bfd990d3dc5f44dff8194f5c33a30516f977bcae6615a8fd8ad206d", "id": "B5.Q1a", "kind": "str", "mistakes": {"3888407bcd4f66824bda54d8a4c8d8961382b75fce8bf171aaf51b1dbdaf97b8": "pour générer, part-on d'une image du dataset ? Relis l'encadré 🔗 de 205.1", "98560c3d2eac11c494f5e3ab63b7ff5060503afc39b75132d37d247589eaf757": "relis l'encadré 🔗 « Lien avec les ch. 25 et 27 » : combien de passes du décodeur pour générer avec un VAE ?"}}
📝 Ex B5.Q1a : réponse enregistrée (str).
WB_ANSWER {"hash": "9765d05e0491668deecbfcfe28e7f8ae69a00f1227d851ef1965a5d5264a0205", "id": "B5.Q1b", "kind": "bool", "mistakes": {"060620eacdb00f1abfbb5d99e1d32d54aa87c468130a034cf0422a3ea62c1812": "relis « Détruire est facile » (§205.1) : le processus direct s'apprend-il ?"}}
📝 Ex B5.Q1b : réponse enregistrée (bool).
WB_ANSWER {"hash": "583732202f06286a76edfdd7cd55f3dc520de1afb3832471b931864676c232c5", "id": "B5.Q1c", "kind": "str", "mistakes": {"226a1167aa75d3e143a25ed867bb40659b37e20fae774013b

**B5.Q2 — Pourquoi détruire l'image en mille petits pas de bruit plutôt qu'en un seul ?**

In [4]:
wb.record("B5.Q2a", "B", mistakes={"relis l'encadré 🧮 « la MSE apprend une moyenne » (§205.1) : que rend un réseau entraîné par MSE quand plusieurs cibles sont possibles ?": "A", "un réseau entraîné par MSE recopie-t-il un exemple ? Relis l'encadré 🧮 de 205.1": "C"})
wb.record("B5.Q2b", Q2B, decimals=2, mistakes={"c'est la médiane : la MSE est minimisée par une autre grandeur (encadré 🧮 de 205.1)": 0.0,
          "c'est le milieu entre la plus petite et la plus grande cible : relis l'encadré 🧮 de 205.1": 0.5})
wb.record("B5.Q2c", Q2C, decimals=2, mistakes={"c'est la MSE de la prédiction 0 : prends la prédiction de b)": 5 / 3,
          "la MSE est une moyenne des carrés des écarts : divise par le nombre de cibles": 3 * Q2C,
          "élève les écarts au carré avant de faire la moyenne": float(np.mean(np.abs(np.array([-1, 0, 2]) - Q2B)))})
wb.record("B5.Q2d", True, mistakes={"relis « Ce qu'on voudrait » (§205.3) : que dit-on de ce pas quand β_t est petit ?": False})
wb.record("B5.Q2e", False, mistakes={"relis l'approximation ln ᾱ_T ≈ −Σβ (§205.2) : à somme égale, que deviennent les β quand il y a deux fois plus de pas ?": True})

WB_ANSWER {"hash": "094818d2a8a2354a0889983c7f1314fcaa69b394cf1774d3bca1958ccf8f9dc8", "id": "B5.Q2a", "kind": "str", "mistakes": {"b4f01a88802c1bcda9330e0f6e43afa29ced09409eb664cf85f62546f88969fc": "un réseau entraîné par MSE recopie-t-il un exemple ? Relis l'encadré 🧮 de 205.1", "c304df0643bf9b28c3a048ca9bee692254b6886c130132a69f34af724f1a9378": "relis l'encadré 🧮 « la MSE apprend une moyenne » (§205.1) : que rend un réseau entraîné par MSE quand plusieurs cibles sont possibles ?"}}
📝 Ex B5.Q2a : réponse enregistrée (str).
WB_ANSWER {"decimals": 2, "hash": "78107d440a9641580dd6fb5192b8db493bc66a5a4adb90cd7e448e056a02678a", "hash_coarse": "a1f4b98c299bedbb6d28b80d7407a26cde30ebffa5e8a43264123ec38c41e3a9", "id": "B5.Q2b", "kind": "float", "magnitude": -1, "mistakes": {"05831900b805231409287bc8e58bbec9763f9d50ba81095ef2f5588975d79921": "c'est le milieu entre la plus petite et la plus grande cible : relis l'encadré 🧮 de 205.1", "bb0f542c4a5e90f59dc4f87d31fcad0f0d4738cf05d68e7a13a2d213b8c

**B5.Q3 — ᾱ_t : quelle part de l'image reste-t-il au pas t ?**

In [5]:
wb.record("B5.Q3a", Q3A, decimals=2, mistakes={"c'est ᾱ_t : relis la forme fermée, le coefficient de x₀ est-il ᾱ_t ?": 0.64,
          "relis la question : on demande le coefficient de x₀, pas celui du bruit": Q3B})
wb.record("B5.Q3b", Q3B, decimals=2, mistakes={"c'est 1 − ᾱ_t : relis la forme fermée, le coefficient du bruit est-il 1 − ᾱ_t ?": 0.36,
          "relis la question : on demande le coefficient du bruit, pas celui de x₀": Q3A})
wb.record("B5.Q3c", False, mistakes={"calcule la somme de tes réponses a) et b) ; relis la dernière phrase de « La forme fermée »": True})
wb.record("B5.Q3d", Q3D, decimals=2, mistakes={"le SNR met la part de l'image au numérateur": 0.36 / 0.64,
          "le SNR est un rapport de variances, pas de coefficients": Q3A / Q3B})
wb.record("B5.Q3e", "C", mistakes={"relis la définition de ᾱ_t : un produit de nombres plus petits que 1 peut-il croître ?": "A", "regarde la figure `plannings.png` (§205.2) : ᾱ_t reste-t-il constant ?": "B"})

WB_ANSWER {"decimals": 2, "hash": "fad54c90c758d36d3f929e8f0beacc3e97d2de79178e24f59997151aab2e57ad", "hash_coarse": "98f03cce9a77a0dd7bd4fd37cf10094a7aa488f9099b214d928cd80ff1c03f4e", "id": "B5.Q3a", "kind": "float", "magnitude": -1, "mistakes": {"5b082ddfe2aa19d480987580180299cf94991e4d7a84ccc7a23f07aa87f0c559": "c'est ᾱ_t : relis la forme fermée, le coefficient de x₀ est-il ᾱ_t ?", "821dde2d176f9264898828ea23fc23e4b682e26de8059af02e2c168104d6d065": "relis la question : on demande le coefficient de x₀, pas celui du bruit"}, "sign": 1}
📝 Ex B5.Q3a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "25c6c597d65c25763d0625df0ff5ea63983e70adcca46841b6518cad4e722314", "hash_coarse": "ef22d2133a9888058e66f00cb45d8e46529016d0478d14c16c49f7015d0a11cd", "id": "B5.Q3b", "kind": "float", "magnitude": -1, "mistakes": {"6f6f88f03b8feffdc7148c78da558fbe48985a87911d928b3f6299a8fe3656ac": "relis la question : on demande le coefficient du bruit, pas celui de x₀", "947b55c

**B5.Q4 — Prédire le bruit, l'image ou la vitesse : la même information ?**

In [6]:
wb.record("B5.Q4a", "A", mistakes={"relis « Prédire le bruit » (§205.3) : la formule de x̂₀ est-elle une approximation quand le bruit est le vrai ?": "B", "relis « Prédire le bruit » (§205.3) : la forme fermée s'inverse-t-elle ?": "C"})
wb.record("B5.Q4b", Q4B, decimals=3, mistakes={"relis la formule de x̂₀ : on divise par la racine de ᾱ_t, et le bruit est multiplié par la racine de 1 − ᾱ_t": (0.5 - 0.36 * 0.2) / 0.64,
          "vérifie le signe : on retire le bruit de x_t": (0.5 + 0.6 * 0.2) / 0.8,
          "n'oublie pas de diviser par la racine de ᾱ_t": 0.5 - 0.6 * 0.2})
wb.record("B5.Q4c", True, mistakes={"relis le paragraphe qui suit la moyenne μ_θ (§205.3) : qu'ont trouvé Ho et coll. pour la prédiction de x₀ ?": False})
wb.record("B5.Q4d", "C", mistakes={"le réseau reçoit-il encore t quand il prédit le bruit ? Relis « Dire au réseau quel est le pas »": "A", "le bruit est-il toujours plus petit que l'image ? Relis la raison donnée en 205.3": "B"})
wb.record("B5.Q4e", False, mistakes={"relis le paragraphe qui suit l'encadré 🧮 sur Euler (§205.9) : les deux « v » sont-ils la même grandeur ?": True})

WB_ANSWER {"hash": "bfb6795642a6d6b50d64e3a40e8b3bb2d4fff9eaa5ee18f857e3bd76cb9c2863", "id": "B5.Q4a", "kind": "str", "mistakes": {"5a67a8bc195589b301e6e26ef998d5504f369e0cc0e83294614b9e4bd7855dd7": "relis « Prédire le bruit » (§205.3) : la formule de x̂₀ est-elle une approximation quand le bruit est le vrai ?", "afc7240a09d1900a530ba6801608680382c0f9ba4e916dcafdb92b7158e9f41a": "relis « Prédire le bruit » (§205.3) : la forme fermée s'inverse-t-elle ?"}}
📝 Ex B5.Q4a : réponse enregistrée (str).
WB_ANSWER {"alt_coarse": ["0ea6fb4152e772a877405d93fc9c9e5fa97403121242285b6e4949464d41d0e6"], "decimals": 3, "hash": "178742548729aa10e29912caf40ed673ca98d46c5b2df4373d38b54f6e2658b8", "hash_coarse": "8cf32c251715715e3549a6e671549d4b8a0e1b0d5f9aef859cfaed5c2eb6ed08", "id": "B5.Q4b", "kind": "float", "magnitude": -1, "mistakes": {"4a7f5bc959d6dd62cc18c51bc96a037d6ea2557f8198c380af127800b6a3be88": "n'oublie pas de diviser par la racine de ᾱ_t", "59248d3c9978905b14f5f3c063f7654c5a7cb203ce53f7fc134

**B5.Q5 — Le pas de temps est une entrée du réseau : pourquoi, et comment l'encoder ?**

In [7]:
wb.record("B5.Q5a", "C", mistakes={"relis « Reconstruire s'apprend » (§205.1) : combien de réseaux sert-on pour tous les pas ?": "A", "ᾱ_t se calcule-t-il dans le réseau ? Relis « Dire au réseau quel est le pas »": "B"})
wb.record("B5.Q5b", False, mistakes={"relis « Dire au réseau quel est le pas » (§205.4) : que dit la fiche d'un nombre de 0 à 999 donné tel quel ?": True})
wb.record("B5.Q5c", Q5C, mistakes={"c'est le nombre de colonnes : chaque couple en occupe deux": 128,
          "relis l'appel : d_model = 128": 32})
wb.record("B5.Q5d", "B", mistakes={"relis « Un bloc résiduel conditionné par le pas » : l'encodage est-il concaténé ?": "A", "relis la ligne `h = h + self.time(temb)[:, :, None, None]` du bloc résiduel": "C"})
wb.record("B5.Q5e", True, mistakes={"relis la fin de « Dire au réseau quel est le pas » : que dit la fiche des permutations de colonnes ?": False})

WB_ANSWER {"hash": "e1cc67682c4ff50a34a369e5128e3a9665034f8b5268bfed646fed0b86f6579e", "id": "B5.Q5a", "kind": "str", "mistakes": {"3e32a67cdd1a6d965cce6563c49ea7ff7bd229844195bb2bdf58803b0d526c3e": "relis « Reconstruire s'apprend » (§205.1) : combien de réseaux sert-on pour tous les pas ?", "45829bd96fc3743ec3ba777759e74f54f1bc8f1cb9d3deb584d9d287d6e0ab5f": "ᾱ_t se calcule-t-il dans le réseau ? Relis « Dire au réseau quel est le pas »"}}
📝 Ex B5.Q5a : réponse enregistrée (str).
WB_ANSWER {"hash": "ce2341a06111b77bf7037f83d0b975ccb798cbdd77b1b857fb96ecd4f34ede85", "id": "B5.Q5b", "kind": "bool", "mistakes": {"f138295139b743639d4b2842c931c2ace22fafedf4e5ce134c3f71ac1160e34b": "relis « Dire au réseau quel est le pas » (§205.4) : que dit la fiche d'un nombre de 0 à 999 donné tel quel ?"}}
📝 Ex B5.Q5b : réponse enregistrée (bool).
WB_ANSWER {"hash": "55b7f4d34aef60e4bc7f5d0432c0f8fd824b93333cd689cd5b189b683c121cdf", "id": "B5.Q5c", "kind": "int", "magnitude": 1, "mistakes": {"4dff560bed34a

**B5.Q6 — U-Net : à quoi servent les connexions de saut ?**

In [8]:
wb.record("B5.Q6a", "A", mistakes={"relis « Le U-Net et ses connexions de saut » : à quelle résolution relient-elles l'encodeur et le décodeur ?": "B", "c'est une connexion résiduelle : relis le piège classique « deux skip connections »": "C"})
wb.record("B5.Q6b", Q6B, mistakes={"la connexion de saut concatène : les canaux de l'encodeur s'ajoutent à ceux du décodeur": 128,
          "relis la question : 128 canaux du décodeur et 64 de l'encodeur": 256,
          "relis la question : la convolution suivante reçoit aussi les canaux du décodeur": 64})
wb.record("B5.Q6c", True, mistakes={"relis le paragraphe qui suit la figure du U-Net (§205.4) : que deviendrait la sortie sans connexions de saut ?": False})
wb.record("B5.Q6d", Q6D, mistakes={"recompte les réductions de l'énoncé : en as-tu appliqué une de moins ?": 8,
          "tu n'as appliqué qu'une réduction : relis l'énoncé": 16})
wb.record("B5.Q6e", "C", mistakes={"la group norm a-t-elle des paramètres γ et β ? Relis le compte du bloc résiduel": "A", "relis la définition de la group norm (§205.4) : que normalise-t-elle ?": "B"})

WB_ANSWER {"hash": "2af279216aa4a1eef516d317f75367fb5e479522b1d2c3d0fc1b487909017dcb", "id": "B5.Q6a", "kind": "str", "mistakes": {"6d4d83e07df204a3c3dd4288ecbf5971f835988e0091319bd523677ba5543401": "relis « Le U-Net et ses connexions de saut » : à quelle résolution relient-elles l'encodeur et le décodeur ?", "afc15161be5ba7bee0d167a69393dee56a41290a24cfe634b94f90254a57baa6": "c'est une connexion résiduelle : relis le piège classique « deux skip connections »"}}
📝 Ex B5.Q6a : réponse enregistrée (str).
WB_ANSWER {"hash": "3243595f7f596899ce48af140a9797adab88a4f0af263ea1f4d6d1a252f21ec7", "id": "B5.Q6b", "kind": "int", "magnitude": 2, "mistakes": {"0f43e0f63f59276edc86ad96d545c78d3cf70945276c5ed8841ffe154bca8b53": "relis la question : 128 canaux du décodeur et 64 de l'encodeur", "8a45ccb2f94a9d2e59cabfb9e208586511e0fc80bd7c2ae5fa42487d5eecec37": "la connexion de saut concatène : les canaux de l'encodeur s'ajoutent à ceux du décodeur", "ecdd1eebfcdaaedd99443d79bdbd5c7f964a3faa14d66017e60

**B5.Q7 — Échantillonnage DDPM : pourquoi est-il lent et aléatoire ?**

In [9]:
wb.record("B5.Q7a", round(Q7A), mistakes={"c'est le nombre de passes : on demande des secondes": 1000,
          "vérifie la multiplication : as-tu décalé la virgule ?": 500})
wb.record("B5.Q7b", False, mistakes={"relis « Ce que coûte un échantillon » (§205.5) : que tire-t-on à chaque pas ?": True})
wb.record("B5.Q7c", True, mistakes={"relis la phrase qui suit l'algorithme 2 (§205.5)": False})
wb.record("B5.Q7d", "A", mistakes={"le planning interdit-il les batches ? Relis « Ce que coûte un échantillon »": "B", "un réseau peut-il servir plusieurs calculs ? Relis « Ce que coûte un échantillon » : de quoi chaque passe a-t-elle besoin ?": "C"})
wb.record("B5.Q7e", "B", mistakes={"relis « Quelle variance ? » (§205.5) : le réseau prédit-il la variance ?": "A", "relis la fin de 205.3 : β̃_t ≤ β_t, et l'égalité ne vaut qu'au premier pas ?": "C"})

WB_ANSWER {"hash": "7abd9a8d854c2571a3934d519e409161e7ed3d2d53f9a0ce9e20f92cc5f3058c", "id": "B5.Q7a", "kind": "int", "magnitude": 1, "mistakes": {"a6e14655a09094448176c4cb5e2331c1360d2038ca64ee7acad944d59f4fd436": "vérifie la multiplication : as-tu décalé la virgule ?", "e6ed7d79d6ba9dcc4dbe2b5d2d89f01e7592f7120e910779d8d60bac6b2795a9": "c'est le nombre de passes : on demande des secondes"}, "sign": 1}
📝 Ex B5.Q7a : réponse enregistrée (int).
WB_ANSWER {"hash": "eaab222072e0334799831a31a53e4751c5b86148ad42644a57743c84093a759e", "id": "B5.Q7b", "kind": "bool", "mistakes": {"618ff13307c3a4a4de193992c5d98993f0993d3bb9db842ad3c43c0096610dad": "relis « Ce que coûte un échantillon » (§205.5) : que tire-t-on à chaque pas ?"}}
📝 Ex B5.Q7b : réponse enregistrée (bool).
WB_ANSWER {"hash": "4454b8b8d94464306b0d6fa11a15d819550b38912c8ed57bdd29efecd705d9a4", "id": "B5.Q7c", "kind": "bool", "mistakes": {"1a2e40b0ed14ed8c03529106a3a38687d1fe8d8e918819ab195ea53875b8c4f3": "relis la phrase qui suit l'

**B5.Q8 — DDIM : moins de pas avec le même réseau, vrai ou faux**

In [10]:
wb.record("B5.Q8a", False, mistakes={"relis « L'idée de DDIM » (§205.6) : l'entraînement change-t-il ?": True})
wb.record("B5.Q8b", True, mistakes={"relis la formule DDIM avec η = 0 : y a-t-il un tirage ?": False})
wb.record("B5.Q8c", True, mistakes={"relis « Le paramètre η » (§205.6) : que vaut σ² quand η = 1 et t' = t − 1 ?": False})
wb.record("B5.Q8d", True, mistakes={"relis l'étape 2 de DDIM (§205.6) : t' doit-il être le pas juste avant ?": False})
wb.record("B5.Q8e", False, mistakes={"relis « Pourquoi c'est plus rapide » (§205.6) : une trajectoire se suit-elle aussi bien avec très peu de pas ?": True})

WB_ANSWER {"hash": "4da32600807556aa9d50ffea57d2f49e09818c57f0a821202c78ebb248a8a8e8", "id": "B5.Q8a", "kind": "bool", "mistakes": {"c5da9d1db4071e5f788099ca426d88f231f2539222a298e5f11335ac70664a55": "relis « L'idée de DDIM » (§205.6) : l'entraînement change-t-il ?"}}
📝 Ex B5.Q8a : réponse enregistrée (bool).
WB_ANSWER {"hash": "01cefb5acadf0ba72942e087616cb638480f47471af7a121d262af006e8a2de9", "id": "B5.Q8b", "kind": "bool", "mistakes": {"94ffca2e98cd92f7b4bfa09853240e7b84068f0978b91758d230edfff0dcbeab": "relis la formule DDIM avec η = 0 : y a-t-il un tirage ?"}}
📝 Ex B5.Q8b : réponse enregistrée (bool).
WB_ANSWER {"hash": "80da9ea4e24f2e5b587aa87232a1c245752fc4b4676f6dba3b01f6ddeab92f94", "id": "B5.Q8c", "kind": "bool", "mistakes": {"5ea00f61d1ae74d9b7205097a9b4e932af46a2b20355e6c6ec8db00502755274": "relis « Le paramètre η » (§205.6) : que vaut σ² quand η = 1 et t' = t − 1 ?"}}
📝 Ex B5.Q8c : réponse enregistrée (bool).
WB_ANSWER {"hash": "1b89795826b2a5f2beea7e1ded6887b1cfd84d805119a

**B5.Q9 — Guidance sans classifieur : ce que règle l'échelle w**

In [11]:
wb.record("B5.Q9a", Q9A, decimals=1, mistakes={"vérifie le signe de l'extrapolation": -Q9A,
          "(1 + w) multiplie la prédiction avec condition, w celle sans condition": 3 * 0.5 - 2 * 0.3,
          "relis la question : w = 2": 0.3})
wb.record("B5.Q9b", "B", mistakes={"relis la formule avec w = 0 : quelle prédiction reste-t-il ?": "A", "relis la formule avec w = 0 : y a-t-il une moyenne ?": "C"})
wb.record("B5.Q9c", False, mistakes={"relis le paragraphe qui suit la figure de la guidance (§205.7)": True})
wb.record("B5.Q9d", 2, mistakes={"relis le piège classique « deux conventions pour l'échelle de guidance » (§205.7)": 3,
          "guidance_scale = 1 + w : résous pour w": 4})
wb.record("B5.Q9e", "C", mistakes={"relis « La guidance sans classifieur » : combien de réseaux entraîne-t-on ?": "A", "le réseau peut-il prédire sans condition s'il n'a jamais vu ∅ ? Relis « La guidance sans classifieur »": "B"})

WB_ANSWER {"decimals": 1, "hash": "0af48be07708958405ce64cf7f9de3610a3d19de99c05f7e84a524e0a223a910", "hash_coarse": "17533303c3aa63bf1014df2b19154c8e0e0ede98269b51a9915f2206b2122eb4", "id": "B5.Q9a", "kind": "float", "magnitude": -1, "mistakes": {"9c38cdc50d8af29fa76486ca47706708d368295bf58d384216b314081a1a5c08": "relis la question : w = 2", "d4faca14fc495a4a7adbe5a2a4cde6be696778f638667e31fcd5eadea2cb8942": "vérifie le signe de l'extrapolation", "e71ede53935594605b7002922965cbf8e2e048c17dfbdf7a50e06fa115c6a15f": "(1 + w) multiplie la prédiction avec condition, w celle sans condition"}, "sign": -1}
📝 Ex B5.Q9a : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"hash": "fc0809dbd7bd694ed8b8d16252c8223fc7ba317291e35cc2e63f2dde8db86f75", "id": "B5.Q9b", "kind": "str", "mistakes": {"a150331c475c472bd4285172026be460da0700cbc96ec9493c4f80b3ee551acc": "relis la formule avec w = 0 : quelle prédiction reste-t-il ?", "e0bbbac766ded1f58fbad31a2f8bf296ee4d323cb828ecc7cb6da333474511cd": "rel

**B5.Q10 — Un bon FID peut-il cacher de la copie des données d'entraînement ?**

In [12]:
wb.record("B5.Q10a", True, mistakes={"relis « La mémorisation » (§205.8) : que verrait une distance de Fréchet d'un modèle qui recopie ?": False})
wb.record("B5.Q10b", "A", mistakes={"plus d'images rend-il la distance de Fréchet sensible à la copie ? Relis « La mémorisation »": "B", "la loss dit-elle quelles images sont recopiées ? Relis « La mémorisation »": "C"})
wb.record("B5.Q10c", False, mistakes={"relis la dernière puce des résultats de Carlini et coll. (§205.8)": True})
wb.record("B5.Q10d", "B", mistakes={"relis les trois règles de « La distance de Fréchet » (§205.8)": "A", "la distance dépend-elle du nombre d'images de façon proportionnelle ? Relis les trois règles de 205.8": "C"})
wb.record("B5.Q10e", False, mistakes={"relis « Fidélité et diversité » (§205.8) : un seul nombre sépare-t-il les deux qualités ?": True})

WB_ANSWER {"hash": "11f4e64b983a4ca64c809ecf789230b362c61ae9d79aa239d4a9553fa4c5352e", "id": "B5.Q10a", "kind": "bool", "mistakes": {"5e726cfd17b47f2d011e4970cc5364b4c129d93e4e48e49ad044a3dc4ee3486f": "relis « La mémorisation » (§205.8) : que verrait une distance de Fréchet d'un modèle qui recopie ?"}}
📝 Ex B5.Q10a : réponse enregistrée (bool).
WB_ANSWER {"hash": "7b2d919e66fdd1c265c90a03ac4e21d35e093ad3d08a838df0ecff069bd8f47c", "id": "B5.Q10b", "kind": "str", "mistakes": {"ab8afd626935256f6c7dc30db96b5806bc1cb0cf5fd1e520b58a7aa0ec8fb372": "la loss dit-elle quelles images sont recopiées ? Relis « La mémorisation »", "e215f80387ac93053f3578231a064482fc765154e7a01d08a94d08e6fa13feb4": "plus d'images rend-il la distance de Fréchet sensible à la copie ? Relis « La mémorisation »"}}
📝 Ex B5.Q10b : réponse enregistrée (str).
WB_ANSWER {"hash": "821c814fedf24a9d0200dd5d6790a12db607a7b739dd9b14cb4ed3f5279cc1a8", "id": "B5.Q10c", "kind": "bool", "mistakes": {"02d0e2a5fa036f7fde108662747407673d

**B5.Q11 — Diffusion latente, transformers de diffusion, flow matching : ce qui a changé depuis 2020**

In [13]:
wb.record("B5.Q11a", Q11A, mistakes={"n'oublie pas les 3 canaux du latent": (256 // 4) ** 2,
          "relis l'énoncé : le latent a 3 canaux": (256 // 4) ** 2 * 4,
          "le facteur f divise la largeur et la hauteur, chacune": 256 * 256 * 3 // 4})
wb.record("B5.Q11b", "A", mistakes={"relis « Les transformers de diffusion » (§205.9) : d'où viennent les tokens d'un DiT ?": "B", "relis « Les transformers de diffusion » : combien de tokens pour un latent de 32 × 32 × 4 ?": "C"})
wb.record("B5.Q11c", "C", mistakes={"dérive x_s par rapport à s : vérifie le signe": "A", "dérive x_s par rapport à s : le coefficient de x₀ est 1 − s": "B"})
wb.record("B5.Q11d", True, mistakes={"relis l'encadré 🧮 « la méthode d'Euler » (§205.9) : d'où vient l'erreur d'Euler ?": False})
wb.record("B5.Q11e", False, mistakes={"relis « Les transformers de diffusion » et « Flow matching et rectified flow » (§205.9)": True})

WB_ANSWER {"hash": "de96e14b3f214c5c80b2f82bc5c815e0ed292442779bc290eb2ee79bc69d1d6b", "id": "B5.Q11a", "kind": "int", "magnitude": 4, "mistakes": {"18e86e4853606208274e2971b5014738375953086fd6a797314118e18bbf9da7": "relis l'énoncé : le latent a 3 canaux", "5f9fc6ffe61cb280aa8d448fbf094f0feabd9fed327f270a5791a599f096fb80": "n'oublie pas les 3 canaux du latent", "c6b4a99e948a5108a410a9a3379362f3e6c636093f116c304adf31343db52a40": "le facteur f divise la largeur et la hauteur, chacune"}, "sign": 1}
📝 Ex B5.Q11a : réponse enregistrée (int).
WB_ANSWER {"hash": "92353b2a193a5a3de8a50def67158fa4ce80a0c71d3596b70471b8631cbd53d8", "id": "B5.Q11b", "kind": "str", "mistakes": {"e33d29a407f8fcf9a138894cf7ea3af1f3f4808d124ac4a13fedcc172b233c15": "relis « Les transformers de diffusion » (§205.9) : d'où viennent les tokens d'un DiT ?", "f1b52e9f20587d428e885f2e8b20a36557a1bc04a4ab091326a402e2f24bd1f8": "relis « Les transformers de diffusion » : combien de tokens pour un latent de 32 × 32 × 4 ?"}}
📝 E

**B5.R1 — L'encodage sinusoïdal des positions**

In [14]:
wb.record("B5.R1a", 1, mistakes={"la colonne 1 est impaire : sinus ou cosinus ?": 0})
wb.record("B5.R1b", R1B, decimals=2, mistakes={"une période de sinus est un tour complet : relis la formule de la colonne 0": math.pi,
          "relis la formule : à la colonne 0, la fréquence vaut 1": 1.0})
wb.record("B5.R1c", False, mistakes={"relis l'encodage sinusoïdal en B3 : quelles colonnes changent vite d'une position à la suivante ?": True})
wb.record("B5.R1d", True, mistakes={"calcule la période 2π · 10 000^(126/128) de la dernière paire et compare-la à 1 000": False})
wb.record("B5.R1e", "B", mistakes={"relis la formule de l'énoncé : PE[p, 2i] est-il un cosinus ?": "A", "relis la formule de l'énoncé": "C"})

WB_ANSWER {"hash": "202d43d8d1c269bb9e12d78036ab5f808798ab51438e19bb4f3660119673093b", "id": "B5.R1a", "kind": "int", "magnitude": 0, "mistakes": {"c84fc6e33ed6f157a8b2f8f6c243c822cfd54635e73af0f6a9da333b1661b944": "la colonne 1 est impaire : sinus ou cosinus ?"}, "sign": 1}
📝 Ex B5.R1a : réponse enregistrée (int).
WB_ANSWER {"decimals": 2, "hash": "6d3933b0ed309f90a2b3b717d8a9549f18e8e27dd36fbebd78d20dd7714eba6a", "hash_coarse": "39a734eef1f0c33882483056bb9048911eb7ff24c729c08f5ac3f8aa6b0fbbab", "id": "B5.R1b", "kind": "float", "magnitude": 0, "mistakes": {"6530bbdb6de830448c4b5982c0260de9cb3447d3df4aff1d403523e01dbd13b1": "une période de sinus est un tour complet : relis la formule de la colonne 0", "d45c99575cc61fd84c83f38d3e968293ba7cfc9bc169d97839c5499aa31e0c6f": "relis la formule : à la colonne 0, la fréquence vaut 1"}, "sign": 1}
📝 Ex B5.R1b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"hash": "212a846eeb25c4f27c7fdd4aecff2bdb6d6e1c331d715b3fd547d5f80e065a37", "id": 

**B5.R2 — Ce que mesure la distance de Fréchet et l'effondrement de mode**

In [15]:
wb.record("B5.R2a", round(R2A), mistakes={"il manque un des deux termes de la formule : la moyenne ou la covariance": 1,
          "n'oublie pas le terme −2(Σ₁Σ₂)^(1/2)": 6},
          fractional="relis le calcul : avec ces nombres, le résultat est entier")
wb.record("B5.R2b", "A", mistakes={"la distance de Fréchet juge-t-elle chaque image à part ? Relis la formule": "B", "une distance nulle demanderait la même moyenne et la même covariance : est-ce le cas ?": "C"})
wb.record("B5.R2c", True, mistakes={"chaque terme de la formule est-il nul quand les moyennes et les covariances sont égales, et seulement alors ?": False})
wb.record("B5.R2d", False, mistakes={"relis le ch. 27 : sur quoi le FID calcule-t-il ses moyennes et covariances ?": True})
wb.record("B5.R2e", round(R2E), mistakes={"un générateur qui rend toujours la même feature a-t-il la covariance des vraies images ?": 0},
          fractional="relis le calcul : avec ces nombres, le résultat est entier")

WB_ANSWER {"fractional": "relis le calcul : avec ces nombres, le résultat est entier", "hash": "b83d9d99b3eed1583b2eaa5e72d559a9750b87880502487d824686e070b64f89", "id": "B5.R2a", "kind": "int", "magnitude": 0, "mistakes": {"61b9200131427d9b18dc01905732e1dabfe69943e249762a18f16f5858b34723": "il manque un des deux termes de la formule : la moyenne ou la covariance", "8b7f9a80672acd1ff7d4e7c15c6d0713e3654cc17c6cb47cf2021cc9041e86d6": "n'oublie pas le terme −2(Σ₁Σ₂)^(1/2)"}, "sign": 1}
📝 Ex B5.R2a : réponse enregistrée (int).
WB_ANSWER {"hash": "11cc26a8c983eac465f3bb2e44c43ae71a9968feebf6067c81917a05612dd26d", "id": "B5.R2b", "kind": "str", "mistakes": {"b4ac2d11ebe3fa70c291149a9e7db70fa53d79b44857ddaa0e5469ac5908269a": "la distance de Fréchet juge-t-elle chaque image à part ? Relis la formule", "d55440c668e38f4ee072c9da2837bf60c89c82879cb7cdc85bd414fcb74d5286": "une distance nulle demanderait la même moyenne et la même covariance : est-ce le cas ?"}}
📝 Ex B5.R2b : réponse enregistrée (st

**B5.R3 — L'astuce de reparamétrisation et le terme KL du VAE**

In [16]:
wb.record("B5.R3a", R3A, decimals=2, mistakes={"ln σ² = ln 0,25 : σ est la racine de 0,25, pas 0,25": 0.5 + 0.25 * (-2),
          "z = μ + σ ε, avec σ = exp(0,5 ln σ²), pas ln σ²": 0.5 + math.log(0.25) * (-2),
          "vérifie le signe : ε = −2": 0.5 + 0.5 * 2})
wb.record("B5.R3b", R3B, decimals=4, mistakes={"n'oublie pas le facteur 1/2 de la formule": 2 * R3B,
          "il manque le terme − ln σ²": 0.5 * (0.25 + 0.25 - 1),
          "σ² = 0,25 : relis le terme de variance de la formule": 0.5 * (0.25 + 0.5 - 1 - math.log(0.25))})
wb.record("B5.R3c", "A", mistakes={"la reparamétrisation change-t-elle la taille du latent ? Relis le ch. 25": "B", "z reste-t-il aléatoire avec la reparamétrisation ? Relis le ch. 25": "C"})
wb.record("B5.R3d", True, mistakes={"relis l'encadré 🔗 « Lien avec la reparamétrisation (ch. 25) » (§205.3)": False})
wb.record("B5.R3e", False, mistakes={"relis « Le lien avec l'ELBO » (§205.3) : ce terme dépend-il des paramètres du réseau ?": True})

WB_ANSWER {"decimals": 2, "hash": "74d92e69dd584e8cd12f4067743354cb01abc0f6d2d58915f7641709340bb60b", "hash_coarse": "c2c7317387fe4e7a5fb26410de3d5731c4fac5fd5e9c51c62f17aa940bd86618", "id": "B5.R3a", "kind": "float", "magnitude": -1, "mistakes": {"1d0008746d8f4c5f9108f2fd031efbcc97b6e531e17de8d2302bb26206b5ab93": "z = μ + σ ε, avec σ = exp(0,5 ln σ²), pas ln σ²", "27633c0bbbf1269b90dd5d644b3efc9e2d7f1c6ec1db27a153bf4cb9e7e45855": "vérifie le signe : ε = −2", "ea380cbd28f2038e3a9b0895ecf26daccbf07fc6f1238f5671a0b0e9e9dbb475": "ln σ² = ln 0,25 : σ est la racine de 0,25, pas 0,25"}, "sign": -1}
📝 Ex B5.R3a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "73a66e9612aa81154a4233733a66e20d6aedcf4e7e1426ac29e9ae347dc48788", "hash_coarse": "bcbf3cb8f72efc38a22c0897eae033cec3ebdc3f73ca5e2c6da87e5206c6d433", "id": "B5.R3b", "kind": "float", "magnitude": -1, "mistakes": {"42a511f3dce60b38f25218a5da75f01c30f8a09a3798e77c5cd80ae3d3dc2382": "σ² = 0,25 : relis le term

**Ex B5.1 — Guidance sans classifieur : combiner les deux prédictions pour w = 0, 1 et 3**

In [17]:
wb.record("B5.1a", W0_B51, decimals=2, mistakes={"avec w = 0, quelle prédiction garde-t-on ? Relis la formule": [0.5, 0.1]})
wb.record("B5.1b", W1_B51, decimals=2, mistakes={"la guidance n'est pas une moyenne des deux prédictions : relis la formule": [0.65, -0.2],
          "(1 + w) multiplie la prédiction avec condition, w celle sans condition": cfg_b51(1, EU_B51, EC_B51),
          "relis la question : w = 1": W0_B51})
wb.record("B5.1c", W3_B51, decimals=2, mistakes={"(1 + w) multiplie la prédiction avec condition, w celle sans condition": cfg_b51(3, EU_B51, EC_B51),
          "c'est guidance_scale = 3, la convention de diffusers : relis le piège classique de 205.7": cfg_b51(2),
          "relis la question : w = 3": W1_B51})
wb.record("B5.1d", 4, mistakes={"relis le piège classique « deux conventions pour l'échelle de guidance » (§205.7)": 3,
          "guidance_scale = 1 + w : vérifie le sens": 2})
wb.record("B5.1e", GAP_B51, decimals=2, mistakes={"multiplie l'écart des deux prédictions par w": GAP_B51 / 3,
          "la norme euclidienne est la racine de la somme des carrés": 3 * (0.3 + 0.6),
          "relis la question : on demande l'écart à ε_c, pas la norme de ε̃": float(np.linalg.norm(W3_B51))})
wb.record("B5.1f", True, mistakes={"remplace w par −1 dans la formule": False})
wb.record("B5.1g", False, mistakes={"compare ta réponse c) aux deux prédictions, composante par composante": True})
wb.record("B5.1h", EVALS_B51, mistakes={"la guidance demande deux prédictions à chaque pas": 50,
          "relis la question : 50 pas DDIM, pas 1 000": 2 * 1000})

WB_ANSWER {"decimals": 2, "element_coarse": [["75cf984b49cac05a94877637e73315d1f2b7acb186378e629665822ae4e69360"], ["4c4a8c4935ee5e1688ff9bb447e876a80f1621d62a739f2c36bc4d1185efb371"]], "element_hashes": ["e48c8f6ba388968e4973462718ee5d4aa31092e4bd8c6e20540f8238a78e2fa8", "41899b55e99828d02fa01a26c79f06fbd3040fffa196695c0050cfa18045eb1d"], "hash": "6e866ad36d5037bc93a7bc059fb047ede01d463e310e7f1dfd1951ca4d3933e6", "id": "B5.1a", "kind": "array", "mistakes": {"106644b23b35aef1a281a8e3c919cee7aba0b75bcb7205b695e108021794f509": "avec w = 0, quelle prédiction garde-t-on ? Relis la formule"}, "shape": [2]}
📝 Ex B5.1a : réponse enregistrée (array, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "element_coarse": [["a4383bfebb4e94ea8d1a966b683350d9973383d74a5d5e3a69ba3de2b0c448ad"], ["8158bbaa05b412dcf247b5884c199e3aa865b6dbbd030c62038f59ce6d83335e"]], "element_hashes": ["2e2ba7cfbaa38dd9ee4198e91d1585521df08c06ca908fb7d1bd3f9472480b2b", "f1e56ab2a569ed3610ac0e5335cc9e5602fb463114da88659bc861d0ad5f

**Ex B5.2 — Planning linéaire : β_t, α_t et ᾱ_t pour T = 4, puis ᾱ_T pour T = 1 000**

In [18]:
wb.record("B5.2a", BETA2_B52, decimals=2, mistakes={"4 valeurs de 0,05 à 0,35 font 3 intervalles, pas 4": 0.05 + 0.3 / 4,
          "relis la question : β_2 est la deuxième valeur, à l'indice 1": 0.25})
wb.record("B5.2b", AB2_B52, decimals=4, mistakes={"c'est α_2 : ᾱ_2 est le produit des α jusqu'au pas 2": float(ALPHAS_B52[1]),
          "ᾱ est un produit des 1 − β, pas 1 moins la somme des β": 1 - 0.05 - 0.15,
          "relis la question : on demande ᾱ_2, au pas 2": float(AB_B52[0])})
wb.record("B5.2c", AB4_B52, decimals=4, mistakes={"c'est α_4 : ᾱ_4 est le produit des α jusqu'au pas 4": float(ALPHAS_B52[3]),
          "ᾱ est un produit des 1 − β, pas 1 moins la somme des β": 1 - float(BETAS_B52.sum()),
          "on demande ᾱ_4, pas sa racine": KEEP4_B52,
          "relis la question : le produit va jusqu'au pas 4": float(AB_B52[2])})
wb.record("B5.2d", KEEP4_B52, decimals=4, mistakes={"c'est ᾱ_4 : relis la forme fermée, le coefficient de x₀ est sa racine": AB4_B52,
          "relis la question : on demande le coefficient de x₀, pas celui du bruit": math.sqrt(1 - AB4_B52)})
wb.record("B5.2e", APPROX4_B52, decimals=4, mistakes={"recompte la somme des quatre β": math.exp(-1.0),
          "relis la question : on demande l'approximation e^(−Σβ), pas la valeur exacte": AB4_B52})
wb.record("B5.2f", False, mistakes={"compare tes réponses c) et e) : calcule l'écart relatif": True})
wb.record("B5.2g", SUM1000_B52, decimals=2, mistakes={"la somme d'une suite arithmétique est T × (premier + dernier) / 2 : n'oublie pas le / 2": 2 * SUM1000_B52,
          "relis la question : on demande la somme des 1 000 β, pas leur moyenne": SUM1000_B52 / 1000,
          "relis l'énoncé : le dernier β vaut 0,02": 1000 * (1e-4 + 0.2) / 2})
wb.record("B5.2h", AB1000_E5_B52, decimals=2, mistakes={"garde la somme des β de g) sans l'arrondir à 10": math.exp(-10) * 1e5,
          "relis la question : multiplie l'approximation par 10⁵": math.exp(-SUM1000_B52) * 1e6,
          "tu as arrondi e^(−Σβ) à cinq décimales : pour un si petit nombre, garde cinq chiffres significatifs": 4.0,
          "c'est le produit exact des 1 − β : on demande l'approximation e^(−Σβ)": float(np.prod(1 - np.linspace(1e-4, 0.02, 1000))) * 1e5})
wb.record("B5.2i", KEEP1000_PCT_B52, decimals=2, mistakes={"h) est multiplié par 10⁵ : reviens à ᾱ_T avant de prendre la racine": math.sqrt(AB1000_E5_B52),
          "relis la question : en pourcentage, donc multiplié par 100": KEEP1000_PCT_B52 / 100,
          "c'est la racine du produit exact : l'énoncé demande celle de ta valeur de h)": 100 * math.sqrt(float(np.prod(1 - np.linspace(1e-4, 0.02, 1000))))})
wb.record("B5.2j", SUM400_B52, decimals=2, mistakes={"relis la question : les mêmes bornes, mais 400 β seulement": SUM1000_B52,
          "n'oublie pas le / 2 de la somme d'une suite arithmétique": 2 * SUM400_B52})
wb.record("B5.2k", SUM400_SCALED_B52, decimals=2, mistakes={"relis la question : les deux bornes sont multipliées par 1 000/400": SUM400_B52,
          "n'oublie pas le / 2 de la somme d'une suite arithmétique": 2 * SUM400_SCALED_B52})
wb.record("B5.2l", True, mistakes={"compare tes réponses g) et k)": False})

WB_ANSWER {"alt_coarse": ["224a543eff99f0b2de7bf76237bf384820558b603d655abee3b6462dae59851c"], "decimals": 2, "hash": "0067a79e6beaa07629aadd1f114b3ff1667466986f35e8acc37a14973db47afd", "hash_coarse": "3203b3757c6cf58604daee14f407b1d9abad07671320857c638a2f548d2d79d5", "id": "B5.2a", "kind": "float", "magnitude": -1, "mistakes": {"459c5919e1341d81e56adbd78bcee8bfbe7056be11b506c4f83bc7922c520517": "4 valeurs de 0,05 à 0,35 font 3 intervalles, pas 4", "e8f110db21a2238bbfc72f982bdc259c846e622a974a01d5118f5a791c5150d1": "relis la question : β_2 est la deuxième valeur, à l'indice 1"}, "sign": 1}
📝 Ex B5.2a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"alt_coarse": ["6872f72db8799cddf431a33ff0e055a559a2ce08f8e6cd4ca4507d20960c907f"], "decimals": 4, "hash": "9439a649f005fc82c5c9e8494be5f16627692d820f33798209d6e1e50b7bf9fb", "hash_coarse": "01f5fe76506d62f4e45227ff7b9f2b29d730d2bbee6cbee429736b53930a3d63", "id": "B5.2b", "kind": "float", "magnitude": -1, "mistakes": {"4e42f9ef93bcbb

**Ex B5.3 — Rapport signal sur bruit : à quel pas un chiffre devient-il illisible, planning linéaire ou cosinus ?**

In [19]:
wb.record("B5.3a", S500_B53, decimals=3, mistakes={"relis la formule de S(t) : t(t − 1), pas t²": lin_sum_b53(500, square=True),
          "S(500) somme les 500 premiers β, pas 500 fois la moyenne de tous les β": 500 * (1e-4 + 0.02) / 2})
wb.record("B5.3b", LIN500_B53, decimals=2, mistakes={"le SNR met la part de l'image au numérateur, celle du bruit au dénominateur": 1 / LIN500_B53,
          "c'est ᾱ_t : on demande le SNR": math.exp(-S500_B53)})
wb.record("B5.3c", COS500_B53, decimals=2, mistakes={"n'oublie pas le décalage s = 0,008": 1.0,
          "c'est ᾱ_t : on demande le SNR": cos_ab_b53(0.5)})
wb.record("B5.3d", LIN1_B53, decimals=2, mistakes={"SNR = 1 veut dire ᾱ = 1/2 : relis le lien entre ᾱ_t et S(t)": lin_cross_b53(1.0),
          "résous l'équation du second degré en t, puis divise t par T": lin_cross_b53(math.log(2)) / 2})
wb.record("B5.3e", COS1_B53, decimals=2, mistakes={"SNR = 1 veut dire ᾱ = 1/2 : c'est le carré du cosinus qui vaut 1/2, pas le cosinus": cos_cross_b53(0.25)})
wb.record("B5.3f", LIN01_B53, decimals=2, mistakes={"SNR = 0,1 ne veut pas dire ᾱ = 0,1 : écris ᾱ en fonction du SNR": lin_cross_b53(math.log(10)),
          "relis la question : le seuil de lisibilité est 0,1, pas 1": LIN1_B53})
wb.record("B5.3g", COS01_B53, decimals=2, mistakes={"n'oublie pas le décalage s = 0,008": cos_cross_b53(1 / 11, s=0.0),
          "SNR = 0,1 ne veut pas dire ᾱ = 0,1 : écris ᾱ en fonction du SNR": cos_cross_b53(0.1)})
wb.record("B5.3h", True, mistakes={"compare ta réponse f) à 0,5": False})
wb.record("B5.3i", False, mistakes={"relis ta réponse g) : quelle part des pas reste lisible ?": True})
wb.record("B5.3j", "B", mistakes={"compare f) − d) et g) − e)": "A", "calcule f) − d) et g) − e) : sont-elles égales ?": "C"})

WB_ANSWER {"decimals": 3, "hash": "dd7f468dd0741eb01bea3e073c31c08efb73870a9bef0a0c04f36c160e2a9967", "hash_coarse": "1c37021d581d21d3cfa20600546e8e9ef41077389f44c3d1a63467885aefcd1b", "id": "B5.3a", "kind": "float", "magnitude": 0, "mistakes": {"51bee9f92e760812f6f084a3e1376839b39818a4e850b142591fd5f8abb0574a": "S(500) somme les 500 premiers β, pas 500 fois la moyenne de tous les β", "e43b033db3a967db4cc93430fa90ad0e55dfe1adf7c97410024c9c2ebd54899f": "relis la formule de S(t) : t(t − 1), pas t²"}, "sign": 1}
📝 Ex B5.3a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "04bbe4817ed33ec112f2ddac10cf9e65fab01cdd297639bdfb7a3ce12778f980", "hash_coarse": "d2d92be9a99273785043510db2fcb695c4aef2a7d671b5fa97053db07e235c0f", "id": "B5.3b", "kind": "float", "magnitude": -2, "mistakes": {"3302ca29d0b6cc1a975e5561e4359b75a9e66b7c2c4c7a4b4568fd0b47efd47c": "le SNR met la part de l'image au numérateur, celle du bruit au dénominateur", "b5fe43e0d0ad9bc6c704d7984f00d509b

**Ex B5.4 — Un pas DDPM à la main, en dimension 1, avec un bruit prédit donné**

In [20]:
wb.record("B5.4a", COEF_B54, decimals=4, mistakes={"relis l'algorithme 2 : β_t est divisé par la racine de 1 − ᾱ_t": float(BETAS_B52[2] / (1 - AB_B52[2])),
          "décalage d'un cran : le pas 3 est l'indice 2, avec β_3 = 0,25": float(BETAS_B52[3] / math.sqrt(1 - AB_B52[3])),
          "relis l'algorithme 2 : c'est 1 − ᾱ_t, pas 1 − α_t": float(BETAS_B52[2] / math.sqrt(BETAS_B52[2]))})
wb.record("B5.4b", MEAN_B54, decimals=4, mistakes={"vérifie le signe : on retire le bruit prédit": ddpm_mean_b54(X3_B54, EPS_B54, sign=1.0),
          "relis l'algorithme 2 : β_t est divisé par la racine de 1 − ᾱ_t": ddpm_mean_b54(X3_B54, EPS_B54, root=False),
          "on divise par la racine de α_t, pas de ᾱ_t": float((X3_B54 - COEF_B54 * EPS_B54) / math.sqrt(AB_B52[2])),
          "décalage d'un cran : le pas 3 est l'indice 2": ddpm_mean_b54(X3_B54, EPS_B54, i=3)})
wb.record("B5.4c", BTILDE_B54, decimals=4, mistakes={"c'est β_3 : la variance du posterior a un facteur en plus": float(BETAS_B52[2]),
          "décalage d'un cran : le pas 3 est l'indice 2": posterior_b54(3)[2],
          "relis la formule de β̃ : (1 − ᾱ_{t−1}) au numérateur, (1 − ᾱ_t) au dénominateur": float(BETAS_B52[2] * (1 - AB_B52[2]) / (1 - AB_B52[1]))})
wb.record("B5.4d", X2_B54, decimals=4, mistakes={"z multiplie l'écart-type σ_t, pas la variance": MEAN_B54 + BTILDE_B54 * Z_B54,
          "vérifie le signe de z : z = −1": MEAN_B54 - math.sqrt(BTILDE_B54) * Z_B54,
          "relis la question : on demande x₂, avec le tirage z": MEAN_B54})
wb.record("B5.4e", X2_BETA_B54, decimals=4, mistakes={"z multiplie l'écart-type, la racine de β_3": MEAN_B54 + float(BETAS_B52[2]) * Z_B54,
          "relis la question : la variance est β_3, pas β̃_3": X2_B54})
wb.record("B5.4f", CLIP_MEAN_B54, decimals=4, mistakes={"tu n'as pas écrêté x̂₀ dans [−1, 1]": NOCLIP_MEAN_B54,
          "relis la question : on demande la moyenne, pas x̂₀ écrêté": 1.0,
          "as-tu arrondi les coefficients du posterior à quatre décimales ? Garde-en cinq : ce chemin tombe à la limite d'un arrondi": 1.2036})
wb.record("B5.4g", LAST_B54, decimals=4, mistakes={"vérifie le signe : on retire le bruit prédit": ddpm_mean_b54(0.5, 0.2, i=0, sign=1.0),
          "relis la question : au pas 1, on prend les constantes de l'indice 0": ddpm_mean_b54(0.5, 0.2, i=1)})
wb.record("B5.4h", False, mistakes={"relis la docstring de ddpm_step (« no noise at all at t = 0 ») et la phrase qui suit l'algorithme 2": True})
wb.record("B5.4i", True, mistakes={"compare tes calculs de d) et e) : quelle partie change ?": False})

WB_ANSWER {"decimals": 4, "hash": "b1e75ea809fcf0089334134f40817cc8f813abba329c7baecc46d356c13b9ad1", "hash_coarse": "c1dc85a0a41fd480635175e25d63361feeddebc1fe87a53020edd5da86116bff", "id": "B5.4a", "kind": "float", "magnitude": -1, "mistakes": {"416e7ffe47ce4ca821a6d3123ddbdc82478cc2f144f319d67d780281e71523ac": "décalage d'un cran : le pas 3 est l'indice 2, avec β_3 = 0,25", "e298d2288abdd80e6bd119148fa06c7a596902f9e2d3048954be750e33222199": "relis l'algorithme 2 : β_t est divisé par la racine de 1 − ᾱ_t", "e9b7d9fdac10783a34838fae264a93a91e675870940439f4c5ee1739d6657c1b": "relis l'algorithme 2 : c'est 1 − ᾱ_t, pas 1 − α_t"}, "sign": 1}
📝 Ex B5.4a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "9c9d979f59e6a7964e192195fb30317e78a22e05688afe8e36b1d8f49bdc1aaa", "hash_coarse": "6da5fde0c481cec3270b44eea159b2aacd31d02f536d3f466d79d0be3f3c5a7b", "id": "B5.4b", "kind": "float", "magnitude": -1, "mistakes": {"2734739da7f52805c415d17ddd157a521d73b2066efb83da

**Ex B5.5 — Un pas DDIM à la main et l'image estimée x̂_0**

In [21]:
wb.record("B5.5a", X0HAT_B55, decimals=4, mistakes={"vérifie le signe : on retire le bruit de x_t": x0hat_b55(eps=-EPS_B55),
          "relis la formule de x̂₀ : des racines de ᾱ_t et de 1 − ᾱ_t": float((X4_B55 - (1 - AB_B52[3]) * EPS_B55) / AB_B52[3]),
          "décalage d'un cran : le pas 4 est l'indice 3": x0hat_b55(i=2)})
wb.record("B5.5b", X1_B55, decimals=4, mistakes={"le pas DDIM réutilise aussi le bruit prédit, multiplié par la racine de 1 − ᾱ_{t'}": ddim_b55(3, 0, noise_term=False)[0],
          "relis la question : on s'arrête au pas 1 (indice 0), pas à l'image propre": X0HAT_B55,
          "relis la question : du pas 4 au pas 1": X2_B55})
wb.record("B5.5c", "B", mistakes={"ta librairie compte les pas en indices : le pas t est l'indice t − 1": "A", "l'indice −1 désigne l'image propre : relis la question, on va au pas 1": "C"})
wb.record("B5.5d", X2_B55, decimals=4, mistakes={"le pas DDIM réutilise aussi le bruit prédit": ddim_b55(3, 1, noise_term=False)[0],
          "décalage d'un cran : le pas 2 est l'indice 1": ddim_b55(3, 2)[0]})
wb.record("B5.5e", True, mistakes={"recalcule x̂₀ à partir de ton x₂ et du même ε_θ : que trouves-tu ?": False})
wb.record("B5.5f", SIGMA2_43_B55, decimals=4, mistakes={"c'est β_4 : relis la formule de σ dans « Le paramètre η »": float(BETAS_B52[3]),
          "on demande σ², pas σ": math.sqrt(SIGMA2_43_B55),
          "relis la formule de σ : le premier rapport est (1 − ᾱ_{t'}) / (1 − ᾱ_t)": float((1 - AB_B52[3]) / (1 - AB_B52[2]) * (1 - AB_B52[3] / AB_B52[2]))})
wb.record("B5.5g", SIGMA_41_B55, decimals=4, mistakes={"on demande σ, pas σ²": SIGMA_41_B55 ** 2,
          "σ tombe près d'une limite d'arrondi : as-tu arrondi σ² à cinq décimales avant la racine ? Garde-en six": math.sqrt(round(SIGMA_41_B55 ** 2, 5)),
          "relis la question : du pas 4 au pas 1, soit de l'indice 3 à l'indice 0": ddim_b55(3, 1, eta=1.0)[1]})
wb.record("B5.5h", DIR_41_B55, decimals=4, mistakes={"n'oublie pas de retirer σ² sous la racine": math.sqrt(1 - AB_B52[0]),
          "prends la racine de 1 − ᾱ₁ − σ²": float(1 - AB_B52[0] - SIGMA_41_B55 ** 2),
          "1 − ᾱ₁ et σ² sont presque égaux : leur différence a perdu des chiffres ; garde six décimales de σ²": math.sqrt(1 - AB_B52[0] - round(SIGMA_41_B55 ** 2, 5)),
          "tu as repris σ arrondi de g) : 1 − ᾱ₁ − σ² est une petite différence, calcule σ² sans l'arrondir": math.sqrt(1 - AB_B52[0] - round(SIGMA_41_B55, 4) ** 2)})
wb.record("B5.5i", False, mistakes={"compare tes réponses g) et h)": True})

WB_ANSWER {"decimals": 4, "hash": "7fe12fc3d66a21be99ac4f5e3233fcf3eb14c34cd8bf3b18ac2a5177706a8afe", "hash_coarse": "f950b957e41e7dee8b8c72414c25bec1c8a7ec39e0fb0fafde41c45d54169a9f", "id": "B5.5a", "kind": "float", "magnitude": -1, "mistakes": {"388471095fc8b5cd1199a3fd8ed6b29729ed4a3bf4740be2d1e6b7aa20f00bb2": "relis la formule de x̂₀ : des racines de ᾱ_t et de 1 − ᾱ_t", "3b258e8f4dd8a5f9eeb2b432efceced75502dbcf358a3f4ec37c9badf13a48de": "vérifie le signe : on retire le bruit de x_t", "be6e024fb5947aef1ce7596570a4e8c34f4b67e5a2dc9c7d47ff9bab92944782": "décalage d'un cran : le pas 4 est l'indice 3"}, "sign": 1}
📝 Ex B5.5a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "68d0628f68cd26e670461c4323870c5a8c6c6da82b8dbab02438016f0e49304b", "hash_coarse": "f6465323aee8a33550b4c1bb6c9115bb24b9c6b739efead1855857dc56e3480c", "id": "B5.5b", "kind": "float", "magnitude": -1, "mistakes": {"2cd11380b1420e644cee71cd2ff99290f604b69c7aa0ae6a41a05984eaab3c91": "le pas

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer à la main un planning de bruit, ses $\bar\alpha_t$ et son rapport signal sur bruit ?
2. Sais-tu dériver la forme fermée et le posterior d'un pas, puis calculer un pas DDPM et un pas DDIM ?
3. Sais-tu appliquer la guidance sans classifieur et traduire `guidance_scale` en $w$ ?
4. Sais-tu estimer le coût d'un échantillonnage, et ce que gagnent DDIM, la distillation et l'espace latent ?

**Pour aller plus loin** : l'article de Ho, Jain et Abbeel et le billet de L. Weng, cités dans la fiche. La suite : le bonus B6, l'explicabilité et l'équité.

*Les exercices de code B5.12 à B5.31 (mathématiques de ta librairie, jouets 2D, U-Net sur MNIST et Fashion-MNIST) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B5` pour obtenir le notebook complet.*